# V4 Search — Alpha=0.7 + Drift Safeguards + Physical Weights

**V3'ten ogrendiklerimiz:**
- alpha=0.3 kazandi AMA drift korumalari yoktu
- pTM cutoff=0.50 en etkili filtre
- rmsd_init>10A olan adimlar DAIMA kotu (TM<0.27)

**V4'teki yeni korumalar:**
1. `rmsd_init hard cutoff (10A)` — buyuk deplasman → hard reject
2. `best-so-far rollback` — TM %40+ duserse en iyi yapiya geri don
3. `adaptive early stopping` — 3 ardisik TM dususunde pipeline dur

**V4 stratejisi:** Bu korumalarla alpha=0.7 + decay tekrar denenecek.
Buyuk adim = hizli yaklasma, korumalar = drift onleme.

**Fazlar:**
- Phase A: 4 weight preset x 3 threshold x 3 decay = 36 run
- Phase B: 4 physical preset x 4 threshold x 2 decay = 32 run
- Phase C: Final 30-step

**Calistirma:** Cell 1→4 setup, Cell 5+ deneyler

## 1. Environment Setup (bir kere calistir)

In [ ]:
import os, sys, shutil

from google.colab import drive
drive.mount('/content/drive', force_remount=True)

if os.path.exists('/content/ANM-openfold3'):
    shutil.rmtree('/content/ANM-openfold3')
!git clone https://github.com/beratkaskaloglu/ANM-openfold3.git /content/ANM-openfold3

if not os.path.exists('/content/ANM-openfold3/openfold3-repo'):
    !git clone https://github.com/aqlaboratory/openfold-3.git /content/ANM-openfold3/openfold3-repo
    !pip install -e /content/ANM-openfold3/openfold3-repo -q
else:
    try:
        import openfold3
    except ImportError:
        !pip install -e /content/ANM-openfold3/openfold3-repo -q

!pip install -q biopython matplotlib numpy torch pandas

sys.path.insert(0, '/content/ANM-openfold3')
sys.path.insert(0, '/content/ANM-openfold3/openfold3-repo')

os.makedirs(os.path.expanduser('~/.openfold3'), exist_ok=True)

from pathlib import Path
from openfold3.entry_points.parameters import (
    download_model_parameters,
    get_default_checkpoint_dir,
    DEFAULT_CHECKPOINT_NAME,
)
_param_dir = get_default_checkpoint_dir()
_param_dir.mkdir(parents=True, exist_ok=True)
download_model_parameters(_param_dir, DEFAULT_CHECKPOINT_NAME, skip_confirmation=True)

print('Environment setup complete.')

# ════════════════════════════════════════════════════
#  CONFIG — V4
#  V3'ten sabit: ptm_cutoff=0.50
#  V4 yeni: rmsd_init cutoff, rollback, adaptive stop
#  V4 hedef: alpha=0.7 + decay ile calismak
# ════════════════════════════════════════════════════

# ── PDB ────────────────────────────────────────────
INITIAL_PDB = '1AKE'
TARGET_PDB  = '4AKE'
CHAIN_ID    = 'A'

# ── Pipeline ───────────────────────────────────────
N_STEPS              = 20
COMBINATION_STRATEGY = 'autostop'
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# ── OF3 ────────────────────────────────────────────
USE_MSA_SERVER       = True
USE_TEMPLATES        = False
NUM_ROLLOUT_STEPS    = None
NUM_DIFFUSION_SAMPLES = 1

# ── V3'ten sabit (degistirmeyecegiz) ───────────────
CONFIDENCE_PTM_CUTOFF    = 0.50   # V3 Phase 3 kazanani
CONFIDENCE_PLDDT_CUTOFF  = 40.0   # dusuk — composite main gate
CONFIDENCE_RANKING_CUTOFF = 0.10  # dusuk — composite main gate
CONFIDENCE_RG_MAX        = 2.0    # hard reject
CONFIDENCE_RG_MIN        = 0.3
CONFIDENCE_CLASH_REJECT  = True
CONFIDENCE_RMSD_INIT_MAX = 10.0   # V4 yeni: rmsd_init hard cutoff

# ── V4 yeni drift korumalari ──────────────────────
ENABLE_BEST_ROLLBACK     = True   # TM duserse best'e geri don
BEST_ROLLBACK_TM_DROP    = 0.40   # %40 dususte rollback
ENABLE_ADAPTIVE_STOP     = True   # ardisik TM dususte dur
ADAPTIVE_STOP_WINDOW     = 3      # 3 ardisik dusus

# ── Fallback ───────────────────────────────────────
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# ── Autostop ───────────────────────────────────────
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# ── Converter ─────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

print('V4 Config ready.')

In [ ]:
# ════════════════════════════════════════════════════
#  CONFIG — V3
# ════════════════════════════════════════════════════

# ── PDB ────────────────────────────────────────────
INITIAL_PDB = '1AKE'
TARGET_PDB  = '4AKE'
CHAIN_ID    = 'A'

# ── Pipeline (V2'den ogrenilenlerle optimize) ──────
N_STEPS              = 20     # 30->20: son 10 step zaten stall
COMBINATION_STRATEGY = 'autostop'
Z_MIXING_ALPHA       = 0.7
N_ANM_MODES          = 20
N_COMBINATIONS       = 20
MAX_COMBO_SIZE       = 3
DF                   = 0.6
DF_MIN               = 0.3
DF_MAX               = 3.0
ANM_CUTOFF           = 15.0
CONTACT_R_CUT        = 10.0
CONTACT_TAU          = 1.5
Z_DIRECTION          = 'plus'

# ── OF3 ────────────────────────────────────────────
USE_MSA_SERVER       = True
USE_TEMPLATES        = False
NUM_ROLLOUT_STEPS    = None
NUM_DIFFUSION_SAMPLES = 1

# ── V2'den sabit parametreler (degistirmeyecegiz) ──
ALPHA_DECAY          = 0.8     # V2 best
MAX_STALL            = 8       # V2 best
CONFIDENCE_RG_MAX    = 2.0     # V3: sikistirildi — Rg>2.0 hard reject
CONFIDENCE_RG_MIN    = 0.3
CONFIDENCE_CLASH_REJECT = True

# ── Fallback ───────────────────────────────────────
ENABLE_FALLBACK           = True
FALLBACK_EXTENDED_ENABLED = True
AUTOSTOP_FALLBACK_LEVELS  = (0, 1, 4, 9)

# ── Autostop ───────────────────────────────────────
AUTOSTOP_V_MAGNITUDE = 1.0
AUTOSTOP_N_STEPS     = 5000
AUTOSTOP_BACK_OFF    = 2

# ── Converter ─────────────────────────────────────
DRIVE_DIR = '/content/drive/MyDrive/ANM-openfold3/checkpoints/finetune_msa'
CHECKPOINT_SELECTION = 'best_bf_r'

print('Config ready.')

## 3. Converter + PDB + OF3 yukleme

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import json as _json
import numpy as np
import torch
import time
import copy
import pandas as pd

for mod_name in list(sys.modules.keys()):
    if mod_name.startswith('src'):
        del sys.modules[mod_name]

from src.converter import PairContactConverter
from src.mode_drive import ModeDrivePipeline, ModeDriveConfig, compute_rmsd, tm_score
from src.composite_confidence import (
    CompositeWeights, compute_composite, composite_score_from_step,
    WEIGHT_PRESETS, THRESHOLD_PRESETS,
    normalize_ptm, normalize_plddt, normalize_pae, normalize_rg, normalize_contact_recon,
)

# ── Checkpoint ──────────────────────────────────────
history_path = os.path.join(DRIVE_DIR, 'history.json')
best_model_path = os.path.join(DRIVE_DIR, 'best_model.pt')

if CHECKPOINT_SELECTION == 'best_bf_r' and os.path.exists(history_path):
    with open(history_path) as f:
        history = _json.load(f)
    best_bf_r = -1
    best_epoch = -1
    for entry in history:
        bf_r = entry.get('val_bf_pearson', 0)
        if bf_r > best_bf_r:
            best_bf_r = bf_r
            best_epoch = entry['epoch']
    epoch_ckpt = os.path.join(DRIVE_DIR, f'epoch_{best_epoch:04d}.pt')
    CHECKPOINT_PATH = epoch_ckpt if os.path.exists(epoch_ckpt) else best_model_path
    print(f'Best bf_r: {best_bf_r:.4f} at epoch {best_epoch}')
else:
    CHECKPOINT_PATH = best_model_path

converter = PairContactConverter(CHECKPOINT_PATH)
print(f'Converter loaded from {CHECKPOINT_PATH}')

# ── PDB ──────────────────────────────────────────────
from Bio.PDB import PDBParser, PDBList
from Bio.SeqUtils import seq1

def fetch_ca(pdb_id, chain_id):
    parser = PDBParser(QUIET=True)
    pdbl = PDBList()
    pdb_file = pdbl.retrieve_pdb_file(pdb_id, pdir='/tmp/pdb_cache', file_format='pdb')
    structure = parser.get_structure(pdb_id, pdb_file)
    chain = [c for c in structure[0].get_chains() if c.id == chain_id][0]
    residues = [r for r in chain if r.get_id()[0] == ' ' and 'CA' in r]
    coords = torch.tensor([r['CA'].get_vector().get_array() for r in residues], dtype=torch.float32)
    sequence = ''.join(seq1(r.get_resname()) for r in residues)
    return coords, sequence

initial_ca, sequence = fetch_ca(INITIAL_PDB, CHAIN_ID)
target_ca, _ = fetch_ca(TARGET_PDB, CHAIN_ID)
N = initial_ca.shape[0]
print(f'Initial: {INITIAL_PDB} chain {CHAIN_ID}, N={N}')
print(f'Target:  {TARGET_PDB} chain {CHAIN_ID}, N={target_ca.shape[0]}')
print(f'RMSD to target: {compute_rmsd(initial_ca, target_ca):.2f} A')
print(f'TM-score: {tm_score(initial_ca, target_ca):.4f}')

# ── OF3 ──────────────────────────────────────────────
_query_dir = '/content/of3_queries'
os.makedirs(_query_dir, exist_ok=True)
_query = {
    'queries': {
        INITIAL_PDB: {
            'chains': [{
                'molecule_type': 'protein',
                'chain_ids': [CHAIN_ID],
                'sequence': sequence,
            }]
        }
    }
}
_query_path = os.path.join(_query_dir, f'{INITIAL_PDB}.json')
with open(_query_path, 'w') as f:
    _json.dump(_query, f, indent=2)

from src.of3_diffusion import load_of3_diffusion
diffusion_fn, zij_trunk = load_of3_diffusion(
    query_json=_query_path,
    device='cuda' if torch.cuda.is_available() else 'cpu',
    use_msa_server=USE_MSA_SERVER,
    use_templates=USE_TEMPLATES,
    num_rollout_steps=NUM_ROLLOUT_STEPS,
    num_samples=NUM_DIFFUSION_SAMPLES,
)
print(f'zij_trunk shape: {zij_trunk.shape}')

from src.autostop_adapter import StructureContext
_pdb_file = PDBList().retrieve_pdb_file(INITIAL_PDB, pdir='/tmp/pdb_cache', file_format='pdb')
structure_ctx = StructureContext.from_pdb(_pdb_file, chain_id=CHAIN_ID)
print(f'StructureContext: N={structure_ctx.struct.N}')
print('OF3 ready.')

## 4. V4 Composite Score Runner (drift korumalari dahil)

In [ ]:
# ════════════════════════════════════════════════════
#  V4 COMPOSITE SCORE RUNNER
#  V3 runner + drift korumalari:
#  - rmsd_init hard cutoff
#  - best-so-far rollback
#  - adaptive early stopping
# ════════════════════════════════════════════════════

def make_config(
    n_steps=N_STEPS,
    alpha=0.7,
    alpha_decay=0.8,
    max_stall=8,
    ptm_cutoff=CONFIDENCE_PTM_CUTOFF,
    plddt_cutoff=CONFIDENCE_PLDDT_CUTOFF,
    ranking_cutoff=CONFIDENCE_RANKING_CUTOFF,
):
    return ModeDriveConfig(
        n_steps=n_steps,
        combination_strategy=COMBINATION_STRATEGY,
        z_mixing_alpha=alpha,
        n_anm_modes=N_ANM_MODES,
        n_combinations=N_COMBINATIONS,
        max_combo_size=MAX_COMBO_SIZE,
        df=DF, df_min=DF_MIN, df_max=DF_MAX,
        anm_cutoff=ANM_CUTOFF,
        contact_r_cut=CONTACT_R_CUT,
        contact_tau=CONTACT_TAU,
        z_direction=Z_DIRECTION,
        num_diffusion_samples=NUM_DIFFUSION_SAMPLES,
        confidence_ptm_cutoff=ptm_cutoff,
        confidence_plddt_cutoff=plddt_cutoff,
        confidence_ranking_cutoff=ranking_cutoff,
        confidence_rg_max=CONFIDENCE_RG_MAX,
        confidence_rg_min=CONFIDENCE_RG_MIN,
        confidence_clash_reject=CONFIDENCE_CLASH_REJECT,
        confidence_rmsd_init_max=CONFIDENCE_RMSD_INIT_MAX,  # V4: rmsd_init hard cutoff
        enable_confidence_fallback=ENABLE_FALLBACK,
        fallback_extended_enabled=FALLBACK_EXTENDED_ENABLED,
        autostop_v_magnitude=AUTOSTOP_V_MAGNITUDE,
        autostop_n_steps=AUTOSTOP_N_STEPS,
        autostop_back_off=AUTOSTOP_BACK_OFF,
        autostop_fallback_levels=AUTOSTOP_FALLBACK_LEVELS,
        autostop_chain_id=CHAIN_ID,
        rejected_alpha_decay=alpha_decay,
        max_consecutive_rejected=max_stall,
        confidence_warmup_steps=0,
        # V4: drift korumalari
        enable_best_rollback=ENABLE_BEST_ROLLBACK,
        best_rollback_tm_drop=BEST_ROLLBACK_TM_DROP,
        enable_adaptive_stop=ENABLE_ADAPTIVE_STOP,
        adaptive_stop_window=ADAPTIVE_STOP_WINDOW,
    )


def run_v4(
    weights: CompositeWeights,
    threshold: float,
    label: str = '',
    n_steps: int = N_STEPS,
    alpha: float = 0.7,
    alpha_decay: float = 0.8,
    max_stall: int = 8,
    early_abort_steps: int = 5,
    verbose: bool = True,
) -> dict:
    """V4 runner: composite score + drift korumalari.
    
    Drift korumalari ModeDriveConfig uzerinden pipeline'a aktarilir:
    - rmsd_init > 10A -> hard reject
    - best-so-far rollback (TM %40+ dususte)
    - adaptive early stopping (3 ardisik TM dusus)
    """
    cfg = make_config(
        n_steps=n_steps, alpha=alpha,
        alpha_decay=alpha_decay, max_stall=max_stall,
    )
    pipe = ModeDrivePipeline(
        converter=converter, config=cfg,
        diffusion_fn=diffusion_fn,
        structure_ctx=structure_ctx,
    )

    print(f'\n{"="*70}')
    print(f'  {label}')
    print(f'  weights: {weights.as_dict()}')
    print(f'  threshold={threshold:.2f}  alpha={alpha}  decay={alpha_decay}  stall={max_stall}')
    print(f'  safeguards: rmsd_init_max={CONFIDENCE_RMSD_INIT_MAX}  '
          f'rollback={ENABLE_BEST_ROLLBACK}(drop={BEST_ROLLBACK_TM_DROP})  '
          f'adaptive_stop={ENABLE_ADAPTIVE_STOP}(w={ADAPTIVE_STOP_WINDOW})')
    print(f'{"="*70}')

    coords_ca = initial_ca.clone()
    z_current = zij_trunk.clone()
    orig_alpha = alpha
    current_alpha = alpha
    consecutive_rejected = 0

    # Best-so-far tracking (runner seviyesinde de — pipeline.run() icinde de var)
    coords_best = initial_ca.clone()
    z_best = zij_trunk.clone()
    tm_best = 0.0
    best_step_idx = -1
    rollback_count = 0
    accepted_tm_history = []

    step_metrics = []
    t0 = time.time()
    stopped_early = False

    for step_idx in range(n_steps):
        cfg.z_mixing_alpha = current_alpha
        sr = pipe.step_with_fallback(
            coords_ca, initial_ca, z_current,
            prev_rmsd=0.0,
            target_coords=target_ca,
            step_idx=step_idx,
        )

        # Composite score hesapla
        comp_score, comp_detail = composite_score_from_step(sr, weights)

        # Hard reject kontrolleri
        hard_reject = False
        reject_reason = ''
        if sr.rg_ratio is not None and sr.rg_ratio > CONFIDENCE_RG_MAX:
            hard_reject = True
            reject_reason = f'Rg={sr.rg_ratio:.1f}>{CONFIDENCE_RG_MAX}'
        if sr.has_clash and CONFIDENCE_CLASH_REJECT:
            hard_reject = True
            reject_reason = 'clash'
        # V4: rmsd_init hard cutoff
        if sr.rmsd > CONFIDENCE_RMSD_INIT_MAX:
            hard_reject = True
            reject_reason = f'rmsd_init={sr.rmsd:.1f}>{CONFIDENCE_RMSD_INIT_MAX}'

        if hard_reject:
            accepted = False
        else:
            accepted = comp_score >= threshold
            if not accepted:
                reject_reason = f'comp={comp_score:.3f}<{threshold}'

        # Target metrikleri
        rmsd_tgt = compute_rmsd(sr.new_ca, target_ca)
        tm_tgt = tm_score(sr.new_ca, target_ca)

        m = {
            'step': step_idx + 1,
            'accepted': accepted,
            'comp_score': comp_score,
            'rmsd_init': sr.rmsd,
            'rmsd_tgt': rmsd_tgt,
            'tm_tgt': tm_tgt,
            'ptm': sr.ptm,
            'plddt_mean': float(sr.plddt.mean()) if sr.plddt is not None else None,
            'ranking': sr.ranking_score,
            'mean_pae': sr.mean_pae,
            'rg_ratio': sr.rg_ratio,
            'contact_recon': sr.contact_recon,
            'contact_of3': sr.contact_of3,
            'has_clash': sr.has_clash,
            'fallback_level': sr.fallback_level,
            'alpha_used': current_alpha,
            'reject_reason': reject_reason,
            'rollback': False,
            **{f'd_{k}': v for k, v in comp_detail.items()},
        }

        if verbose:
            tag = 'ACCEPT' if accepted else 'REJECT'
            ptm_s = f'{sr.ptm:.3f}' if sr.ptm is not None else '-'
            pae_s = f'{sr.mean_pae:.1f}' if sr.mean_pae is not None else '-'
            rg_s = f'{sr.rg_ratio:.2f}' if sr.rg_ratio is not None else '-'
            extra = f'  {reject_reason}' if not accepted else ''
            print(
                f'  [{step_idx+1:>2}/{n_steps}] {tag:<6}  '
                f'comp={comp_score:.3f}  '
                f'pTM={ptm_s}  mPAE={pae_s}  Rg={rg_s}  '
                f'RMSD_init={sr.rmsd:.1f}  '
                f'RMSD_tgt={rmsd_tgt:.2f}  TM={tm_tgt:.3f}  '
                f'FB={sr.fallback_level}  a={current_alpha:.3f}'
                f'{extra}'
            )

        # Update state
        if accepted:
            coords_ca = sr.new_ca
            z_current = sr.z_modified
            consecutive_rejected = 0
            current_alpha = orig_alpha

            # Best-so-far rollback
            if tm_tgt > tm_best:
                tm_best = tm_tgt
                coords_best = sr.new_ca.clone()
                z_best = sr.z_modified.clone()
                best_step_idx = step_idx
            elif tm_best > 0 and tm_tgt < tm_best * (1.0 - BEST_ROLLBACK_TM_DROP):
                # Drift! Best'e geri don
                if verbose:
                    print(
                        f'  >> ROLLBACK: TM={tm_tgt:.3f} < best={tm_best:.3f}*'
                        f'{1.0-BEST_ROLLBACK_TM_DROP:.2f} — '
                        f'step {best_step_idx+1}\'e geri donuluyor'
                    )
                coords_ca = coords_best.clone()
                z_current = z_best.clone()
                rollback_count += 1
                m['rollback'] = True

            # Adaptive early stopping
            accepted_tm_history.append(tm_tgt)
            if len(accepted_tm_history) >= ADAPTIVE_STOP_WINDOW + 1:
                recent = accepted_tm_history[-(ADAPTIVE_STOP_WINDOW + 1):]
                monoton_azalis = all(
                    recent[i] > recent[i + 1]
                    for i in range(len(recent) - 1)
                )
                if monoton_azalis:
                    if verbose:
                        tm_seq = ' -> '.join(f'{t:.3f}' for t in recent)
                        print(
                            f'  >> EARLY STOP: {ADAPTIVE_STOP_WINDOW} ardisik TM dusus: '
                            f'{tm_seq} — best step {best_step_idx+1} donduruluyor'
                        )
                    stopped_early = True
                    m['early_stop'] = True
                    step_metrics.append(m)
                    break
        else:
            consecutive_rejected += 1
            if alpha_decay < 1.0:
                current_alpha = max(0.02, current_alpha * alpha_decay)
            if max_stall > 0 and consecutive_rejected >= max_stall:
                if verbose:
                    print(f'  STALL: {consecutive_rejected} consecutive rejected')
                break

        step_metrics.append(m)

        # Early abort
        if step_idx + 1 == early_abort_steps:
            n_accepted = sum(1 for m in step_metrics if m['accepted'])
            if n_accepted == 0:
                if verbose:
                    print(f'  EARLY ABORT: 0/{early_abort_steps} accepted in first steps')
                break

    wall = time.time() - t0
    total_steps = len(step_metrics)
    n_accepted = sum(1 for m in step_metrics if m['accepted'])

    # Best result
    accepted_metrics = [m for m in step_metrics if m['accepted']]
    if accepted_metrics:
        best_step = max(accepted_metrics, key=lambda m: m['tm_tgt'])
        best_tm = best_step['tm_tgt']
        best_rmsd = best_step['rmsd_tgt']
    else:
        best_tm = tm_score(initial_ca, target_ca)
        best_rmsd = compute_rmsd(initial_ca, target_ca)

    result = {
        'label': label,
        'threshold': threshold,
        'alpha': alpha,
        'alpha_decay': alpha_decay,
        'total_steps': total_steps,
        'accepted': n_accepted,
        'rejected': total_steps - n_accepted,
        'accept_rate': n_accepted / max(1, total_steps),
        'best_tm': best_tm,
        'best_rmsd': best_rmsd,
        'rollback_count': rollback_count,
        'stopped_early': stopped_early,
        'wall_s': wall,
        'step_metrics': step_metrics,
    }

    if verbose:
        print(f'\n  => {n_accepted}/{total_steps} accepted ({n_accepted/max(1,total_steps)*100:.0f}%)  '
              f'best_TM={best_tm:.4f}  best_RMSD={best_rmsd:.2f}  '
              f'rollbacks={rollback_count}  early_stop={stopped_early}  wall={wall:.0f}s')

    return result


ALL_RESULTS = {}
print('V4 Runner ready.')

## 5. Phase A: Alpha=0.7 + Decay Grid (4 preset x 3 threshold x 3 decay)

Alpha=0.7 ile buyuk adim, 3 farkli decay stratejisi.
Yeni korumalar (rmsd_init cutoff, rollback, adaptive stop) ile drift yonetilebilir mi?

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE A: ALPHA=0.7 + DECAY GRID
#  4 weight preset x 3 threshold x 3 decay = 36 run
#  Yeni drift korumalari aktif
# ════════════════════════════════════════════════════

PHASE_A_PRESETS = ['A_ptm_heavy', 'B_pae_heavy', 'C_balanced', 'D_physical']
PHASE_A_THRESHOLDS = [0.45, 0.50, 0.55]
PHASE_A_DECAYS = [0.8, 0.9, 1.0]

ALL_RESULTS['phase_a'] = []

for wname in PHASE_A_PRESETS:
    weights = WEIGHT_PRESETS[wname]
    for thr in PHASE_A_THRESHOLDS:
        for decay in PHASE_A_DECAYS:
            decay_label = f'd{decay:.1f}'.replace('.', '')
            label = f'{wname}_t{thr:.2f}_{decay_label}'
            r = run_v4(
                weights=weights,
                threshold=thr,
                label=label,
                alpha=0.7,
                alpha_decay=decay,
            )
            ALL_RESULTS['phase_a'].append(r)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase_a']:
    rows.append({
        'label': r['label'],
        'threshold': r['threshold'],
        'decay': r['alpha_decay'],
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'rollbacks': r['rollback_count'],
        'early_stop': r['stopped_early'],
        'wall_s': f"{r['wall_s']:.0f}",
    })
df_a = pd.DataFrame(rows)
print('\n' + '='*90)
print('PHASE A RESULTS — Alpha=0.7 + Decay + Safeguards')
print('='*90)
print(df_a.to_string(index=False))

# En iyi
best_a = max(ALL_RESULTS['phase_a'], key=lambda r: (r['best_tm'], r['accepted']))
print(f'\nBest Phase A: {best_a["label"]} — '
      f'{best_a["accepted"]}/{best_a["total_steps"]} accepted, '
      f'TM={best_a["best_tm"]:.4f}, rollbacks={best_a["rollback_count"]}')

## 6. Phase B: Physical Presets + Alpha=0.7 (4 preset x 4 threshold x 2 decay)

E/F/G strict normalizer presetleri ile Rg-dominant filtreleme.
Alpha=0.7 + decay=0.8/0.9 ile test.

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE B: PHYSICAL PRESETS + ALPHA=0.7
#  4 physical preset x 4 threshold x 2 decay = 32 run
# ════════════════════════════════════════════════════

PHASE_B_PRESETS = ['D_physical', 'E_physical_strict', 'F_physical_balanced', 'G_rg_dominant']
PHASE_B_THRESHOLDS = THRESHOLD_PRESETS_PHYSICAL  # [0.40, 0.45, 0.50, 0.55]
PHASE_B_DECAYS = [0.8, 0.9]

ALL_RESULTS['phase_b'] = []

for wname in PHASE_B_PRESETS:
    weights = WEIGHT_PRESETS[wname]
    for thr in PHASE_B_THRESHOLDS:
        for decay in PHASE_B_DECAYS:
            decay_label = f'd{decay:.1f}'.replace('.', '')
            label = f'{wname}_t{thr:.2f}_{decay_label}'
            r = run_v4(
                weights=weights,
                threshold=thr,
                label=label,
                alpha=0.7,
                alpha_decay=decay,
            )
            ALL_RESULTS['phase_b'].append(r)

# Sonuc tablosu
rows = []
for r in ALL_RESULTS['phase_b']:
    rows.append({
        'label': r['label'],
        'threshold': r['threshold'],
        'decay': r['alpha_decay'],
        'steps': r['total_steps'],
        'accepted': r['accepted'],
        'accept%': f"{r['accept_rate']*100:.0f}%",
        'best_TM': f"{r['best_tm']:.4f}",
        'best_RMSD': f"{r['best_rmsd']:.2f}",
        'rollbacks': r['rollback_count'],
        'early_stop': r['stopped_early'],
        'wall_s': f"{r['wall_s']:.0f}",
    })
df_b = pd.DataFrame(rows)
print('\n' + '='*90)
print('PHASE B RESULTS — Physical Presets + Alpha=0.7 + Safeguards')
print('='*90)
print(df_b.to_string(index=False))

best_b = max(ALL_RESULTS['phase_b'], key=lambda r: (r['best_tm'], r['accepted']))
print(f'\nBest Phase B: {best_b["label"]} — '
      f'{best_b["accepted"]}/{best_b["total_steps"]} accepted, '
      f'TM={best_b["best_tm"]:.4f}, rollbacks={best_b["rollback_count"]}')

## 7. Phase C: Final 30-step (en iyi A+B konfigurasyonu)

In [ ]:
# ════════════════════════════════════════════════════
#  PHASE C: FINAL 30-STEP
#  Phase A ve B'nin en iyileri ile 30 step
# ════════════════════════════════════════════════════

# Phase A en iyi
best_a = max(ALL_RESULTS['phase_a'], key=lambda r: (r['best_tm'], r['accepted']))
# Phase B en iyi
best_b = max(ALL_RESULTS['phase_b'], key=lambda r: (r['best_tm'], r['accepted']))

# Genel en iyi
all_ab = ALL_RESULTS['phase_a'] + ALL_RESULTS['phase_b']
best_overall = max(all_ab, key=lambda r: (r['best_tm'], r['accepted']))

# Label'dan weight name ve parametreleri coz
best_label = best_overall['label']
# label format: PRESETNAME_tTHR_dDECAY
parts = best_label.rsplit('_', 2)
if len(parts) == 3:
    best_wname = parts[0]
    best_thr = best_overall['threshold']
    best_decay = best_overall['alpha_decay']
elif len(parts) == 4:
    best_wname = f'{parts[0]}_{parts[1]}'
    best_thr = best_overall['threshold']
    best_decay = best_overall['alpha_decay']
else:
    # fallback: manual parse
    best_wname = best_label.split('_t')[0]
    best_thr = best_overall['threshold']
    best_decay = best_overall['alpha_decay']

final_weights = WEIGHT_PRESETS[best_wname]

print(f'Phase A best: {best_a["label"]} — TM={best_a["best_tm"]:.4f}')
print(f'Phase B best: {best_b["label"]} — TM={best_b["best_tm"]:.4f}')
print(f'\nFinal config (overall best):')
print(f'  weights: {best_wname} -> {final_weights.as_dict()}')
print(f'  threshold: {best_thr}')
print(f'  alpha: 0.7, decay: {best_decay}')

# 30 step final run
r_final = run_v4(
    weights=final_weights,
    threshold=best_thr,
    label='FINAL_30step',
    n_steps=30,
    alpha=0.7,
    alpha_decay=best_decay,
    max_stall=12,
    early_abort_steps=8,
)
ALL_RESULTS['phase_c'] = [r_final]

print(f'\nFINAL: {r_final["accepted"]}/{r_final["total_steps"]} accepted '
      f'({r_final["accept_rate"]*100:.0f}%), '
      f'best_TM={r_final["best_tm"]:.4f}, best_RMSD={r_final["best_rmsd"]:.2f}, '
      f'rollbacks={r_final["rollback_count"]}, early_stop={r_final["stopped_early"]}')

## 8. Analiz ve Grafikler

In [ ]:
# ════════════════════════════════════════════════════
#  ANALYSIS AND PLOTS — V4
# ════════════════════════════════════════════════════
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rcParams['figure.dpi'] = 120

def plot_phase_comparison(phase_results, title=''):
    fig, axes = plt.subplots(1, 4, figsize=(20, max(6, len(phase_results)*0.3)))

    labels = [r['label'][-30:] for r in phase_results]  # truncate
    accept_rates = [r['accept_rate'] * 100 for r in phase_results]
    tms = [r['best_tm'] for r in phase_results]
    rollbacks = [r.get('rollback_count', 0) for r in phase_results]
    walls = [r['wall_s'] for r in phase_results]

    colors_ar = ['#2ecc71' if ar > 20 else '#e74c3c' for ar in accept_rates]
    axes[0].barh(labels, accept_rates, color=colors_ar)
    axes[0].set_xlabel('Accept Rate (%)')
    axes[0].set_title('Accept Rate')
    axes[0].axvline(20, color='gray', linestyle='--', alpha=0.5)

    colors_tm = ['#3498db' if t > 0.6 else '#e67e22' for t in tms]
    axes[1].barh(labels, tms, color=colors_tm)
    axes[1].set_xlabel('Best TM-score')
    axes[1].set_title('Best TM-score')
    axes[1].axvline(0.6, color='gray', linestyle='--', alpha=0.5)

    axes[2].barh(labels, rollbacks, color='#e74c3c')
    axes[2].set_xlabel('Rollback Count')
    axes[2].set_title('Rollbacks (drift events)')

    axes[3].barh(labels, walls, color='#9b59b6')
    axes[3].set_xlabel('Wall time (s)')
    axes[3].set_title('Wall Time')

    fig.suptitle(title, fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()


def plot_step_trajectory(result, title=''):
    metrics = result['step_metrics']
    steps = [m['step'] for m in metrics]
    comp = [m['comp_score'] for m in metrics]
    tm = [m['tm_tgt'] for m in metrics]
    accepted = [m['accepted'] for m in metrics]
    rollbacks = [m.get('rollback', False) for m in metrics]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

    for s, c, a, rb in zip(steps, comp, accepted, rollbacks):
        if rb:
            color = '#f39c12'  # rollback = turuncu
            marker = 'D'
        elif a:
            color = '#2ecc71'
            marker = 'o'
        else:
            color = '#e74c3c'
            marker = 'o'
        ax1.scatter(s, c, color=color, s=80, zorder=5, marker=marker, edgecolors='black', linewidths=0.5)
    ax1.plot(steps, comp, 'k-', alpha=0.3)
    ax1.axhline(result.get('threshold', 0.50), color='orange', linestyle='--', label='threshold')
    ax1.set_ylabel('Composite Score')
    ax1.set_title(f'{title} — Composite Score (green=accept, red=reject, diamond=rollback)')
    ax1.legend()
    ax1.grid(True, alpha=0.3)

    for s, t, a, rb in zip(steps, tm, accepted, rollbacks):
        if rb:
            color = '#f39c12'
            marker = 'D'
        elif a:
            color = '#2ecc71'
            marker = 'o'
        else:
            color = '#e74c3c'
            marker = 'o'
        ax2.scatter(s, t, color=color, s=80, zorder=5, marker=marker, edgecolors='black', linewidths=0.5)
    ax2.plot(steps, tm, 'k-', alpha=0.3)
    ax2.set_ylabel('TM-score vs target')
    ax2.set_xlabel('Step')
    ax2.set_title('TM-score vs Target')
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()


# Phase A top 10
if 'phase_a' in ALL_RESULTS:
    top_a = sorted(ALL_RESULTS['phase_a'], key=lambda r: -r['best_tm'])[:10]
    plot_phase_comparison(top_a, 'Phase A: Top 10 (Alpha=0.7 + Decay)')

# Phase B top 10
if 'phase_b' in ALL_RESULTS:
    top_b = sorted(ALL_RESULTS['phase_b'], key=lambda r: -r['best_tm'])[:10]
    plot_phase_comparison(top_b, 'Phase B: Top 10 (Physical Presets)')

# Final trajectory
if 'phase_c' in ALL_RESULTS:
    r_final = ALL_RESULTS['phase_c'][0]
    plot_step_trajectory(r_final, 'Final Run (30 step)')

print('Analysis complete.')

## 9. V3 vs V4 Karsilastirmasi

In [ ]:
# ════════════════════════════════════════════════════
#  V3 vs V4 COMPARISON
#  V3 best: alpha=0.3, ptm_cutoff=0.50, A_ptm_heavy
#  V4: alpha=0.7 + decay + safeguards
# ════════════════════════════════════════════════════

# V3 baseline run (alpha=0.3, no decay, same safeguards)
print('Running V3 baseline (alpha=0.3, same safeguards)...')
r_v3 = run_v4(
    weights=WEIGHT_PRESETS['A_ptm_heavy'],
    threshold=0.50,
    label='V3_baseline_a03',
    n_steps=20,
    alpha=0.3,
    alpha_decay=1.0,
)

# V4 final
r_v4 = ALL_RESULTS.get('phase_c', [{}])[0]

print(f'\n{"="*70}')
print(f'{"V3 (alpha=0.3) vs V4 (alpha=0.7) COMPARISON":^70}')
print(f'{"="*70}')
print(f'{"Metric":<25} {"V3 (a=0.3)":>20} {"V4 (a=0.7)":>20}')
print(f'{"-"*70}')
print(f'{"Accept rate":<25} '
      f'{r_v3["accepted"]}/{r_v3["total_steps"]} ({r_v3["accept_rate"]*100:.0f}%):>20 '
      f'{r_v4.get("accepted","?")}/{r_v4.get("total_steps","?")} ({r_v4.get("accept_rate",0)*100:.0f}%):>20')
print(f'{"Best TM-score":<25} {r_v3["best_tm"]:>20.4f} {r_v4.get("best_tm",0):>20.4f}')
print(f'{"Best RMSD (A)":<25} {r_v3["best_rmsd"]:>20.2f} {r_v4.get("best_rmsd",0):>20.2f}')
print(f'{"Rollbacks":<25} {r_v3["rollback_count"]:>20} {r_v4.get("rollback_count",0):>20}')
print(f'{"Early stop":<25} {str(r_v3["stopped_early"]):>20} {str(r_v4.get("stopped_early","")):>20}')
print(f'{"Wall time (s)":<25} {r_v3["wall_s"]:>20.0f} {r_v4.get("wall_s",0):>20.0f}')
print(f'{"="*70}')

ALL_RESULTS['v3_baseline'] = [r_v3]

## 10. Sonuclari Drive'a Kaydet

In [ ]:
# ════════════════════════════════════════════════════
#  SAVE TO DRIVE
# ════════════════════════════════════════════════════
import json
from datetime import datetime

save_dir = '/content/drive/MyDrive/ANM-openfold3/search_v4'
os.makedirs(save_dir, exist_ok=True)
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')

# Serialize
save_data = {}
for phase_name, results in ALL_RESULTS.items():
    if isinstance(results, list):
        save_data[phase_name] = []
        for r in results:
            r_copy = {k: v for k, v in r.items()}
            save_data[phase_name].append(r_copy)
    else:
        save_data[phase_name] = results

# Save JSON
save_path = os.path.join(save_dir, f'results_{timestamp}.json')
with open(save_path, 'w') as f:
    json.dump(save_data, f, indent=2, default=str)
print(f'Results saved to {save_path}')

# Save summary CSV
all_rows = []
for phase_name, results in ALL_RESULTS.items():
    if not isinstance(results, list):
        continue
    for r in results:
        all_rows.append({
            'phase': phase_name,
            'label': r.get('label', ''),
            'threshold': r.get('threshold', ''),
            'alpha': r.get('alpha', ''),
            'decay': r.get('alpha_decay', ''),
            'steps': r.get('total_steps', ''),
            'accepted': r.get('accepted', ''),
            'accept_rate': f"{r.get('accept_rate', 0)*100:.0f}%",
            'best_tm': f"{r.get('best_tm', 0):.4f}",
            'best_rmsd': f"{r.get('best_rmsd', 0):.2f}",
            'rollbacks': r.get('rollback_count', 0),
            'early_stop': r.get('stopped_early', False),
            'wall_s': f"{r.get('wall_s', 0):.0f}",
        })
csv_path = os.path.join(save_dir, f'summary_{timestamp}.csv')
pd.DataFrame(all_rows).to_csv(csv_path, index=False)
print(f'CSV saved to {csv_path}')

print('\nDone! Sonuclari Drive\'dan indir ve analiz et.')